# Emotionale Wörter identifizieren


In [ ]:
# Hilfsfunktionen laden
import sys
sys.path.append('..')
from kurs_helpers import (
    lade_bereinigten_text,
    GERMAN_STOPWORDS,
    GRIMM_SENTIMENT_LEXIKON,
    berechne_sentiment,
    berechne_sentiment_mit_negation,
    pruefe_07_aufgabe_1,
)


Nachdem wir die grundlegenden Funktionen in Python gelernt haben und Vorbereitungen für unsere Analyse getroffen haben, wollen wir nun die Sentimentanalyse anhand des Märchens "Marienkind" durchführen.

Dies ist einfacher, als gedacht, denn wir wissen ja bereits, wie wir Wörter zählen und uns die häufigsten Wörter ausgeben lassen können. Um nun eine emotionale Analyse der Märchen durchzuführen, wollen wir jetzt nicht mehr nur die häufigsten Wörter zählen, sondern nur die "emotionalen" Wörter.

Wir wollen zunächst exemplarisch herausfinden, welchen "Sentiment Score" das Märchen "Marienkind" hat, um anschließend den Sentiment Score mehrerer Märchen miteinander vergleichen zu können.

Hierfür bieten sich **Dictionaries** in Python an.

## Dictionaries

Dictionaries sind ähnlich zu Listen, jedoch besteht jeder Eintrag nicht nur aus dem Wert an sich, sondern aus einem **Schlüssel (key)** und einem **Wert (value)**.
Hinsichtlich der Syntax werden Listen in `[ ]` geschrieben, Dictionaries werden in `{ }` geschrieben.

### Beispiel "Marienkind"

Mit den wichtigsten Daten eines Märchens kann ich z.B. ein Dictionary anlegen. Innerhalb der geschweiften Klammern stehen dabei die Angaben "Schlüssel:Wert".


In [ ]:
maerchen = {
    "titel": "Marienkind",
    "jahr": 1812,
    "ort": "Wald",
    "figur": "das Mädchen"
}


Über den Schlüssel können wir dann den Wert abfragen (vergleichbar mit dem Index in Listen):


In [ ]:
print("Titel:", maerchen["titel"])
print("Schauplatz:", maerchen["ort"])


Als Wert kann ich nicht nur einzelne Werte wie Integer (ganze Zahlen) oder Strings (Zeichenketten) übergeben, sondern auch ganze Listen:


In [ ]:
maerchen = {
    "titel": "Marienkind",
    "jahr": 1812,
    "figuren": ["Holzhacker", "das Mädchen", "Jungfrau Maria", "der König"],
    "ort": "Wald",
    "moral": "Wahre Reue und das Eingeständnis der eigenen Schuld wiegen schwerer als die Sünde selbst."
}

print("Der Titel lautet:", maerchen["titel"])
print("Der Schauplatz ist:", maerchen["ort"])
print("Es spielen einige Figuren mit:", maerchen["figuren"])


Das können wir uns zunutze machen.

## Ein kleines Sentiment-Lexikon

Die Idee der Sentimentanalyse ist einfach: Wir erstellen ein Wörterbuch (Dictionary), in dem emotionale Wörter einen Wert bekommen. **Positive Wörter** erhalten den Wert **+1**, **negative Wörter** erhalten den Wert **-1**. Dann zählen wir, welche dieser Wörter im Märchen vorkommen, und addieren die Werte.

Ein positiver Gesamt-Score bedeutet: Das Märchen hat eine insgesamt positive Stimmung. Ein negativer Score bedeutet: Die Stimmung ist insgesamt negativ.

Beginnen wir mit einem kleinen Lexikon zum Ausprobieren:


In [ ]:
# Ein kleines Demo-Lexikon mit 12 Wörtern
klein_lexikon = {
    "schön": 1, "prachtvoll": 1, "lieb": 1,
    "glück": 1, "küssen": 1, "goldene": 1,
    "böse": -1, "schlecht": -1, "tod": -1,
    "weinen": -1, "furcht": -1, "garstig": -1
}

# Test: Welchen Wert hat "böse" und welchen Wert hat "schön"?
print("Wert für 'böse':", klein_lexikon["böse"])
print("Wert für 'schön':", klein_lexikon["schön"])


Das Lexikon funktioniert im Grunde also wie ein klassisches Wörterbuch: Wir schlagen ein Wort (key) nach und erhalten seine Definition als Wert (value).

### Sentiment Score berechnen — Schritt für Schritt

Bevor wir den Score für ein ganzes Märchen berechnen, üben wir zunächst mit ein paar einzelnen Wörtern. So verstehen wir, was genau passiert.

**Schritt 1:** Prüfen, ob ein einzelnes Wort im Lexikon steht:


In [ ]:
# Ein einzelnes Wort im Lexikon nachschlagen:
if "böse" in klein_lexikon:
    print("'böse' ist im Lexikon!")
    print("Wert:", klein_lexikon["böse"])
else:
    print("'böse' ist nicht im Lexikon.")

print("Welchen Wert hat es?", klein_lexikon["böse"])

print()
print("Ist 'wald' im Lexikon?", "wald" in klein_lexikon)


**Schritt 2:** Mehrere Wörter nacheinander prüfen — mit einer for-Schleife:


In [ ]:
test_woerter = ["schön", "wald", "böse", "mädchen", "glück"]

for wort in test_woerter:
    if wort in klein_lexikon:
        print(wort, "gefunden! Wert:", klein_lexikon[wort])
    else:
        print(wort, " ist nicht im Lexikon — wird übersprungen.")


Wir sehen: Nur die Wörter, die tatsächlich im Lexikon stehen, werden berücksichtigt. Alle anderen werden übersprungen.

**Schritt 3:** Jetzt addieren wir die Werte auf, um einen Gesamt-Score zu berechnen:


In [ ]:
gesamt_score = 0
gefundene_woerter = []

for wort in test_woerter:
    if wort in klein_lexikon:
        wert = klein_lexikon[wort]
        gesamt_score = gesamt_score + wert
        gefundene_woerter.append(wort)

print("Gesamt-Score:", gesamt_score)
print("Gefundene Wörter:", gefundene_woerter)


Genau dieses Vorgehen wenden wir jetzt auf ein ganzes Märchen an.

### Sentiment Score für Marienkind berechnen

Zuerst laden wir den bereinigten Text und filtern die Stoppwörter heraus:


In [ ]:
# Text laden und bereinigen
bereinigter_text, wort_liste = lade_bereinigten_text('Marienkind.txt')

# Stoppwörter und Zahlen entfernen
gefilterte_liste = []
for wort in wort_liste:
    if wort not in GERMAN_STOPWORDS and not wort.isdigit():
        gefilterte_liste.append(wort)

print("Wörter gesamt: ", len(wort_liste))
print("Wörter nach Filterung: ", len(gefilterte_liste))


Jetzt berechnen wir den Score — genau wie eben geübt, nur mit der echten Wortliste:


In [ ]:
gesamt_score = 0
gefundene_woerter = []

for wort in gefilterte_liste:
    if wort in klein_lexikon:
        wert = klein_lexikon[wort]
        gesamt_score = gesamt_score + wert
        gefundene_woerter.append(wort)

print("Sentiment Score (kleines Lexikon):", gesamt_score)
print("Gefundene emotionale Wörter:", gefundene_woerter)


Wir sehen: Mit nur 12 Wörtern im Lexikon finden wir zwar einige emotionale Wörter, aber viele gehen uns verloren. Wörter wie "traurig", "grausam", "glücklich" oder "tapfer" werden nicht erkannt, weil sie nicht in unserem kleinen Lexikon stehen.

---

## Das große Grimm-Sentiment-Lexikon

Für eine aussagekräftigere Analyse brauchen wir ein umfangreicheres Lexikon. Wir haben für diesen Kurs ein spezielles Sentiment-Lexikon erstellt, das auf die Sprache der Grimm'schen Märchen zugeschnitten ist — mit ca. 180 Wörtern.

Dieses Lexikon ist bereits in unseren Hilfsfunktionen hinterlegt. Schauen wir es uns an:


In [ ]:
# Wie viele Wörter enthält das große Lexikon?
positive = []
negative = []

for wort, wert in GRIMM_SENTIMENT_LEXIKON.items():
    if wert > 0:
        positive.append(wort)
    else:
        negative.append(wort)

print("Gesamtzahl Wörter im Lexikon: ", len(GRIMM_SENTIMENT_LEXIKON))
print("Davon positiv: ", len(positive))
print("Davon negativ: ", len(negative))
print()
print("Einige positive Wörter:", positive[:10])
print("Einige negative Wörter:", negative[:10])


Berechnen wir den Sentiment Score von Marienkind jetzt nochmal — mit dem großen Lexikon:


In [ ]:
score_gross, woerter_gross = berechne_sentiment(gefilterte_liste)

print("Sentiment Score (großes Lexikon):", score_gross)
print("Gefundene emotionale Wörter: ", len(woerter_gross))
print()
print("Die gefundenen Wörter:", woerter_gross)


Vergleichen Sie die beiden Ergebnisse: Mit dem kleinen Lexikon (12 Wörter) haben wir nur wenige emotionale Wörter gefunden. Mit dem großen Lexikon (ca. 180 Wörter) bekommen wir ein deutlich differenzierteres Bild.

---

## Negationserkennung

Es gibt aber noch ein Problem: Was passiert mit einem Satz wie *"Er war **nicht** böse"*? In unserer einfachen Zählung würde "böse" als negativ gewertet (-1), obwohl der Satz eigentlich etwas Positives ausdrückt.

Die Lösung: Wir schauen uns die Wörter **vor** einem Sentiment-Wort an. Wenn dort ein Negationswort steht (z.B. "nicht", "kein", "nie"), drehen wir das Vorzeichen um: Aus -1 wird +1 und umgekehrt.

Testen wir das zuerst an einem konstruierten Beispiel:


In [ ]:
# Beispielsatz zum Ausprobieren:
beispiel = "das mädchen war nicht böse und sehr schön".split()
score, details = berechne_sentiment_mit_negation(beispiel)
print("Score:", score)
print()
for wort, originalwert, negiert, endwert in details:
    if negiert:
        print("  '" + wort + "':", originalwert, "→", endwert, "(NEGIERT!)")
    else:
        print("  '" + wort + "':", endwert)
print("Der Gesamtscore ist: ", score)

Sie sehen: "böse" hat normalerweise den Wert -1. Aber "nicht" steht davor — also wird das Vorzeichen umgedreht: -1 wird zu +1.

### Wie funktioniert die Negationserkennung?

Die Funktion schaut sich für jedes Sentiment-Wort die **drei Wörter davor** an. Wenn dort ein Negationswort steht (wie "nicht", "kein", "keine", "nie", "niemals" u.a.), wird das Vorzeichen umgedreht.

Wenden wir die Negationserkennung jetzt auf Marienkind an:


In [ ]:
score_negation, details_negation = berechne_sentiment_mit_negation(gefilterte_liste)

print("Sentiment Score (mit Negationserkennung):", score_negation)
print()

# Zeige Wörter, bei denen die Negation eingegriffen hat
negierte = []
for wort, orig, negiert, end in details_negation:
    if negiert:
        negierte.append((wort, orig, end))

if negierte:
    print("Negation hat folgende Wörter umgedreht:")
    for wort, orig, end in negierte:
        print("  '" + wort + "':", orig, "→", end)
else:
    print("In diesem Märchen wurden keine Negationen vor Sentiment-Wörtern gefunden.")
    print("Das kommt vor — nicht jeder Text enthält negierte emotionale Ausdrücke.")


---

### Hinweis: Grenzen der Sentimentanalyse

Unsere Analyse ist ein guter Einstieg, hat aber Grenzen:

- **Wortformen:** "Königs" oder "Königin" werden nicht erkannt, wenn nur "König" im Lexikon steht. Professionelle Analysen nutzen dafür *Lemmatisierung* (Zurückführung auf die Grundform).
- **Kontext:** "Listig" ist beim *Schneiderlein* eine positive Eigenschaft (Cleverness), bei der *Hexe* in Hänsel und Gretel eher negativ. Unser Lexikon kann diesen Unterschied nicht erfassen.
- **Lexikongröße:** Auch 180 Wörter erfassen nicht alles. Professionelle Lexika enthalten tausende Einträge.

Trotzdem: Für einen ersten Einblick in die emotionale Färbung eines Textes reicht unser Ansatz aus — und wir können die Ergebnisse verschiedener Märchen miteinander vergleichen!

---

## Übung

**Aufgabe 1:** Berechnen Sie den Sentiment Score für "Frau Holle".

Nutzen Sie dafür die gleichen Schritte wie oben:
1. Text laden und bereinigen (mit `lade_bereinigten_text()`)
2. Stoppwörter und Zahlen entfernen (mit einer for-Schleife)
3. Sentiment Score berechnen — einmal ohne und einmal mit Negationserkennung

Speichern Sie den Score (mit Negation) in `score_frauholle` und die gefundenen Wörter in `gefundene_frauholle`.


In [ ]:
# 1. Text laden

# 2. Stoppwörter und Zahlen entfernen
gefilterte_frauholle = []

# 3. Sentiment Score berechnen (ohne Negation)

# 4. Sentiment Score berechnen (mit Negation)


In [ ]:
# Überprüfung:
pruefe_07_aufgabe_1()


---

[← Zurück zur Übersicht](../00_Willkommen.ipynb) | [← Zurück zu Kapitel 6: Abschlussprüfung Teil I](../Teil_1/06_Abschlusspruefung_Teil_I.ipynb) | [Weiter zu Kapitel 8: Sentimentanalyse vertiefen →](08_Sentimentanalyse_vertiefen.ipynb)
